In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os, time, warnings, random
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import r2_score

from sklearn.linear_model import LinearRegression, Ridge, Lasso, HuberRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.svm import SVR

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from scipy.signal import hilbert
from vmdpy import VMD
from PyEMD import EMD, CEEMDAN

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", DEVICE)

Using: cuda


In [2]:
file_path = r"D:\Projects\Python Scripts\AQI 2026\final_data\AQI_Bengaluru_2021_2025.xlsx"

df = pd.read_excel(file_path)
series = df['AQI'].values.astype(float)

n = len(series)
RUNS = 20
lag = 30
horizon = 1
train_size = int(0.6*n)
val_size   = int(0.2*n)

train = series[:train_size]
val   = series[train_size:train_size+val_size]
test  = series[train_size+val_size:]

In [3]:
print("Train:", len(train), "Val:", len(val), "Test:", len(test))

Train: 930 Val: 310 Test: 310


In [4]:
BASE_DIR = r"D:\Projects\Python Scripts\AQI 2026\simple_aqi\\"
a="h"+str(horizon)
folders = [
    "Bengaluru/modes/" + a,
    "Bengaluru/results/" + a + "/forecasts",
    "Bengaluru/results/" + a + "/metrics",
    "Bengaluru/results/" + a + "/logs",
    "Bengaluru/figures/" + a + "/shap",
    "Bengaluru/figures/" + a + "/feature_importance"
]

for f in folders:
    os.makedirs(os.path.join(BASE_DIR, f), exist_ok=True)

In [5]:
def iceemdan(signal, max_imf=10, ensemble_size=50, noise_strength=0.2):

    signal = np.array(signal)
    N = len(signal)

    imfs = []
    residue = signal.copy()

    for _ in range(max_imf):

        ensemble = []

        for _ in range(ensemble_size):

            noise = np.random.normal(0, 1, N)
            noise_scaled = noise_strength * np.std(residue) * noise

            noisy = residue + noise_scaled

            imf = EMD().emd(noisy, max_imf=1)
            if len(imf) > 0:
                ensemble.append(imf[0])

        if len(ensemble) == 0:
            break

        imf_k = np.mean(ensemble, axis=0)
        imfs.append(imf_k)

        residue = residue - imf_k

        if np.std(residue) < 0.01 * np.std(signal):
            break

    imfs.append(residue)
    return np.array(imfs)

In [6]:
def filter_imfs(imfs):
    energy = np.sum(imfs**2, axis=1)
    threshold = 0.01 * np.max(energy)
    return imfs[energy > threshold]


def classify_imfs(imfs):
    high, low = [], []

    for imf in imfs:
        if np.std(imf) > 0.2 * np.std(imfs):
            high.append(imf)
        else:
            low.append(imf)

    return high, low


def apply_vmd(signal, K=4):
    alpha, tau, DC, init, tol = 2000, 0, 0, 1, 1e-7
    u, _, _ = VMD(signal, alpha, tau, K, DC, init, tol)
    return u


def hybrid_iceemdan_vmd(signal, K_total=6):

    imfs = iceemdan(signal)
    imfs = filter_imfs(imfs)

    high, low = classify_imfs(imfs)

    # 🔥 pick strongest high IMFs
    high = sorted(high, key=lambda x: np.std(x), reverse=True)[:2]

    modes = []

    # 🔷 VMD on high IMFs
    for imf in high:
        vmd_modes = apply_vmd(imf, K=3)
        modes.extend(vmd_modes)

    # 🔷 add low-frequency IMFs
    modes.extend(low)

    modes = np.array(modes)

    # =========================================================
    # 🔥 FORCE FIXED NUMBER OF MODES (THIS IS THE KEY FIX)
    # =========================================================
    if modes.shape[0] > K_total:
        modes = modes[:K_total]

    elif modes.shape[0] < K_total:
        pad = np.zeros((K_total - modes.shape[0], modes.shape[1]))
        modes = np.vstack([modes, pad])

    return modes

In [7]:
K = 6  # choose fixed number

train_modes = hybrid_iceemdan_vmd(train, K_total=K)

val_modes_full  = hybrid_iceemdan_vmd(np.concatenate([train, val]), K_total=K)
test_modes_full = hybrid_iceemdan_vmd(np.concatenate([train, val, test]), K_total=K)

val_modes  = val_modes_full[:, -len(val):]
test_modes = test_modes_full[:, -len(test):]

In [8]:
train_modes_scaled = []
val_modes_scaled  = []
test_modes_scaled  = []
mode_scalers = []

for i in range(train_modes.shape[0]):

    scaler = StandardScaler()

    train_m = train_modes[i].reshape(-1,1)
    val_m   = val_modes[i].reshape(-1,1)
    test_m  = test_modes[i].reshape(-1,1)

    train_s = scaler.fit_transform(train_m).flatten()
    val_s   = scaler.transform(val_m).flatten()
    test_s  = scaler.transform(test_m).flatten()

    train_modes_scaled.append(train_s)
    val_modes_scaled.append(val_s)
    test_modes_scaled.append(test_s)
    mode_scalers.append(scaler)

train_modes_scaled = np.array(train_modes_scaled)
val_modes_scaled = np.array(val_modes_scaled)
test_modes_scaled = np.array(test_modes_scaled)

In [9]:
def create_dataset(data, lag=lag, horizon=horizon):

    X, y = [], []

    for i in range(len(data) - lag - horizon):
        X.append(data[i:i+lag])
        y.append(data[i+lag:i+lag+horizon])

    return np.array(X), np.array(y)

In [10]:
def RMSE(y,yhat): return np.sqrt(np.mean((y-yhat)**2))
def MAE(y,yhat): return np.mean(np.abs(y-yhat))
def SMAPE(y,yhat):
    return np.mean(
        np.abs(y - yhat) / ((np.abs(y) + np.abs(yhat)) / 2 + 1e-8)
    )
def MASE(y,yhat): return MAE(y,yhat)/(np.mean(np.abs(np.diff(y)))+1e-8)
def R2(y,yhat): return r2_score(y,yhat)

In [11]:
def add_residual(x, out):
    # x: (B, T, 1)
    # out: (B, H)

    last = x[:, -1, :]  # (B,1)

    if out.shape[1] > 1:
        last = last.repeat(1, out.shape[1])

    return out + last

class LSTMModel(nn.Module):
    def __init__(self, h, hidden=128, dropout=0.3):
        super().__init__()

        self.lstm = nn.LSTM(1, hidden, batch_first=True, num_layers=2)
        self.norm = nn.LayerNorm(hidden)
        self.do   = nn.Dropout(dropout)

        self.head = nn.Sequential(
            nn.Linear(hidden, hidden//2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden//2, h)
        )

    def forward(self, x):
        out, _ = self.lstm(x)

        z = out[:, -1, :]
        z = self.norm(z)
        z = self.do(z)

        out = self.head(z)
        return add_residual(x, out)

class GRUModel(nn.Module):
    def __init__(self, h, hidden=128, dropout=0.3):
        super().__init__()

        self.gru = nn.GRU(
            input_size=1,
            hidden_size=hidden,
            num_layers=2,
            batch_first=True,
            dropout=dropout
        )

        self.norm = nn.LayerNorm(hidden)
        self.do = nn.Dropout(dropout)

        self.head = nn.Sequential(
            nn.Linear(hidden, hidden//2),
            nn.ReLU(),
            nn.Linear(hidden//2, h)
        )

    def forward(self, x):
        out, _ = self.gru(x)
        z = out[:, -1, :]
        z = self.norm(z)
        z = self.do(z)
        out = self.head(z)
        return add_residual(x, out)

class BiLSTMModel(nn.Module):
    def __init__(self, h, hidden=128, dropout=0.3):
        super().__init__()

        self.lstm = nn.LSTM(
                            1, hidden,
                            num_layers=2,
                            batch_first=True,
                            bidirectional=True,
                            dropout=0.2
                        )

        self.norm = nn.LayerNorm(hidden*2)
        self.do   = nn.Dropout(dropout)

        self.head = nn.Sequential(
            nn.Linear(hidden*2, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, h)
        )

    def forward(self, x):
        out, _ = self.lstm(x)

        z = out[:, -1, :]
        z = self.norm(z)
        z = self.do(z)

        out = self.head(z)
        return add_residual(x, out)

class BiGRUModel(nn.Module):
    def __init__(self, h, hidden=128, dropout=0.3):
        super().__init__()

        self.gru = nn.GRU(
            1, hidden, batch_first=True,
            num_layers=2, bidirectional=True
        )

        self.norm = nn.LayerNorm(hidden*2)
        self.do   = nn.Dropout(dropout)

        self.head = nn.Sequential(
            nn.Linear(hidden*2, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, h)
        )

    def forward(self, x):
        out, _ = self.gru(x)

        z = out[:, -1, :]
        z = self.norm(z)
        z = self.do(z)

        out = self.head(z)
        return add_residual(x, out)

class CNNModel(nn.Module):
    def __init__(self, h, channels=128, dropout=0.3):
        super().__init__()

        # self.conv1 = nn.Conv1d(1, channels, 3, padding=1)
        # self.conv2 = nn.Conv1d(channels, channels, 3, padding=1)
        self.conv = nn.Sequential(
                                        nn.Conv1d(1, 64, 3, padding=1),
                                        nn.ReLU(),
                                        nn.Conv1d(64, 128, 3, padding=1),
                                        nn.ReLU(),
                                        nn.AdaptiveAvgPool1d(1)
                                    )

        self.act  = nn.ReLU()
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.do   = nn.Dropout(dropout)

        self.head = nn.Sequential(
            nn.Linear(channels, channels//2),
            nn.ReLU(),
            nn.Linear(channels//2, h)
        )

    def forward(self, x):
        x_perm = x.permute(0,2,1)

        x_perm = self.act(self.conv(x_perm))

        x_perm = self.pool(x_perm).squeeze(-1)
        x_perm = self.do(x_perm)

        out = self.head(x_perm)
        return add_residual(x, out)

class CNNLSTMModel(nn.Module):
    def __init__(self, h, hidden=128, dropout=0.3):
        super().__init__()

        self.conv = nn.Conv1d(1, 64, 3, padding=1)
        self.act  = nn.ReLU()

        self.lstm = nn.LSTM(64, hidden, batch_first=True)

        self.do = nn.Dropout(dropout)

        self.head = nn.Sequential(
            nn.Linear(hidden, hidden//2),
            nn.ReLU(),
            nn.Linear(hidden//2, h)
        )

    def forward(self, x):
        x_perm = x.permute(0,2,1)
        x_perm = self.act(self.conv(x_perm))

        x_perm = x_perm.permute(0,2,1)
        out, _ = self.lstm(x_perm)

        z = out[:, -1, :]
        z = self.do(z)

        out = self.head(z)
        return add_residual(x, out)

class TransformerModel(nn.Module):
    def __init__(self, h, d_model=128, nhead=4, nlayers=3, dropout=0.1):
        super().__init__()

        self.fc_in = nn.Linear(1, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dropout=dropout,
            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(encoder_layer, nlayers)

        self.norm = nn.LayerNorm(d_model)
        self.do   = nn.Dropout(dropout)

        self.head = nn.Sequential(
            nn.Linear(d_model, d_model//2),
            nn.ReLU(),
            nn.Linear(d_model//2, h)
        )

    def forward(self, x):
        x_enc = self.fc_in(x)
        x_enc = self.encoder(x_enc)

        z = x_enc[:, -1, :]
        z = self.norm(z)
        z = self.do(z)

        out = self.head(z)
        return add_residual(x, out)

In [12]:
class DeepLSTMModel(nn.Module):
    def __init__(self, h, hidden=128):
        super().__init__()

        self.lstm = nn.LSTM(
            1, hidden,
            num_layers=3,
            batch_first=True,
            dropout=0.3
        )

        self.head = nn.Linear(hidden, h)

    def forward(self, x):
        out, _ = self.lstm(x)
        z = out[:, -1, :]
        return add_residual(x, self.head(z))

class DeepGRUModel(nn.Module):
    def __init__(self, h, hidden=128):
        super().__init__()

        self.gru = nn.GRU(
            1, hidden,
            num_layers=3,
            batch_first=True,
            dropout=0.3
        )

        self.head = nn.Linear(hidden, h)

    def forward(self, x):
        out, _ = self.gru(x)
        z = out[:, -1, :]
        return add_residual(x, self.head(z))

class CNNGRUModel(nn.Module):
    def __init__(self, h, hidden=128, dropout=0.3):
        super().__init__()

        # 🔷 CNN
        self.conv = nn.Conv1d(1, 64, kernel_size=3, padding=1)
        self.act  = nn.ReLU()

        # 🔷 GRU
        self.gru = nn.GRU(64, hidden, batch_first=True)

        # 🔷 Norm + Dropout
        self.norm = nn.LayerNorm(hidden)
        self.do   = nn.Dropout(dropout)

        # 🔷 Head
        self.head = nn.Sequential(
            nn.Linear(hidden, hidden//2),
            nn.ReLU(),
            nn.Linear(hidden//2, h)
        )

    def forward(self, x):
        x_orig = x  # 🔥 keep original input

        # 🔷 CNN
        x = x.permute(0, 2, 1)       # (B,1,T)
        x = self.act(self.conv(x))   # (B,C,T)
        x = x.permute(0, 2, 1)       # (B,T,C)

        # 🔷 GRU
        out, _ = self.gru(x)
        z = out[:, -1, :]

        # 🔷 Normalize + Dropout
        z = self.norm(z)
        z = self.do(z)

        # 🔷 Prediction
        out = self.head(z)

        # 🔥 CORRECT residual
        return out + x_orig[:, -1, :]

class ResidualLSTMModel(nn.Module):
    def __init__(self, h, hidden=128):
        super().__init__()

        self.lstm1 = nn.LSTM(1, hidden, batch_first=True)
        self.lstm2 = nn.LSTM(hidden, hidden, batch_first=True)

        self.head = nn.Linear(hidden, h)

    def forward(self, x):
        out1, _ = self.lstm1(x)
        out2, _ = self.lstm2(out1)

        z = out2[:, -1, :]
        return add_residual(x, self.head(z))

class AttentionLSTMModel(nn.Module):
    def __init__(self, h, hidden=128):
        super().__init__()

        self.lstm = nn.LSTM(1, hidden, batch_first=True)
        self.attn = nn.Linear(hidden, 1)

        self.head = nn.Linear(hidden, h)

    def forward(self, x):
        out, _ = self.lstm(x)

        w = torch.softmax(self.attn(out), dim=1)
        z = torch.sum(w * out, dim=1)

        return add_residual(x, self.head(z))

class TCNModel(nn.Module):
    def __init__(self, h):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv1d(1, 64, 3, padding=2, dilation=2),
            nn.ReLU(),
            nn.Conv1d(64, 128, 3, padding=4, dilation=4),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)
        )

        self.fc = nn.Linear(128, h)

    def forward(self, x):
        x = x.permute(0,2,1)
        x = self.net(x).squeeze(-1)
        return add_residual(x.unsqueeze(-1), self.fc(x))

class CNNAttentionModel(nn.Module):
    def __init__(self, h, channels=64, dropout=0.3):
        super().__init__()

        # 🔷 CNN
        self.conv = nn.Conv1d(1, channels, kernel_size=3, padding=1)
        self.act  = nn.ReLU()

        # 🔷 Attention projection
        self.attn = nn.Sequential(
            nn.Linear(channels, channels),
            nn.Tanh(),
            nn.Linear(channels, 1)
        )

        # 🔷 Normalization
        self.norm = nn.LayerNorm(channels)

        # 🔷 Dropout
        self.do = nn.Dropout(dropout)

        # 🔷 Head
        self.fc = nn.Sequential(
            nn.Linear(channels, channels//2),
            nn.ReLU(),
            nn.Linear(channels//2, h)
        )

    def forward(self, x):
        # x: (B, T, 1)
        x_orig = x  # 🔥 keep original for residual

        # 🔷 CNN
        x = x.permute(0, 2, 1)        # (B,1,T)
        x = self.act(self.conv(x))    # (B,C,T)
        x = x.permute(0, 2, 1)        # (B,T,C)

        # 🔷 Attention weights
        w = torch.softmax(self.attn(x), dim=1)   # (B,T,1)

        # 🔷 Context vector
        z = torch.sum(w * x, dim=1)              # (B,C)

        # 🔷 Normalize + Dropout
        z = self.norm(z)
        z = self.do(z)

        # 🔷 Prediction
        out = self.fc(z)

        # 🔥 Residual connection (CRITICAL)
        return out + x_orig[:, -1, :]

In [13]:
def train_torch(model, X_train, y_train, X_val, y_val, device,
                epochs=150, lr=5e-4, batch_size=64):

    model = model.to(device)

    X_train = torch.tensor(X_train, dtype=torch.float32).unsqueeze(-1).to(device)
    y_train = torch.tensor(y_train, dtype=torch.float32).to(device)

    X_val = torch.tensor(X_val, dtype=torch.float32).unsqueeze(-1).to(device)
    y_val = torch.tensor(y_val, dtype=torch.float32).to(device)

    train_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(X_train, y_train),
        batch_size=batch_size,
        shuffle=False   # 🔥 FIXED
    )

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, patience=5, factor=0.5
    )

    best_loss = float("inf")
    patience = 12
    counter = 0

    for epoch in range(epochs):

        model.train()
        for xb, yb in train_loader:

            pred = model(xb)

            loss = 1.0*nn.MSELoss()(pred, yb) + 0*nn.L1Loss()(pred, yb)
            weight_decay = 1e-4  # in AdamW

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        # 🔷 VALIDATION
        model.eval()
        with torch.no_grad():
            val_pred = model(X_val)
            val_loss = nn.MSELoss()(val_pred, y_val)

        scheduler.step(val_loss)

        if val_loss < best_loss:
            best_loss = val_loss
            best_state = model.state_dict()
            counter = 0
        else:
            counter += 1

        if counter >= patience:
            break

    model.load_state_dict(best_state)
    return model

In [14]:
class ELMModel:
    def __init__(self, input_size, hidden_size=200, activation='tanh'):
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.activation = activation

    def _act(self, x):
        if self.activation == 'tanh':
            return np.tanh(x)
        elif self.activation == 'relu':
            return np.maximum(0, x)
        else:
            return 1 / (1 + np.exp(-x))

    def fit(self, X, y):
        self.W = np.random.randn(self.input_size, self.hidden_size)
        self.b = np.random.randn(self.hidden_size)

        H = self._act(X @ self.W + self.b)

        # Moore–Penrose pseudo-inverse
        self.beta = np.linalg.pinv(H) @ y

    def predict(self, X):
        H = self._act(X @ self.W + self.b)
        return H @ self.beta

In [15]:
def build_model(mid, h=1, input_size=None):

    if mid==0: return LinearRegression()
    if mid==1: return Ridge()
    if mid==2: return Lasso()
    if mid==3: return HuberRegressor()
    if mid==4: return DecisionTreeRegressor()
    if mid==5: return RandomForestRegressor(n_estimators=200)
    if mid==6: return ExtraTreesRegressor(n_estimators=200)
    if mid==7: return GradientBoostingRegressor()
    if mid==8: return AdaBoostRegressor()
    if mid==9: return SVR()
    if mid==10: return XGBRegressor(tree_method="hist", verbosity=0)
    if mid==11: return LGBMRegressor(verbose=-1)
    if mid==12: return CatBoostRegressor(verbose=0)

    if mid == 13: return LSTMModel(h).to(DEVICE)
    if mid == 14: return GRUModel(h).to(DEVICE)
    if mid == 15: return BiLSTMModel(h).to(DEVICE)
    if mid == 16: return CNNModel(h).to(DEVICE)
    if mid == 17: return CNNLSTMModel(h).to(DEVICE)
    if mid == 18: return TransformerModel(h).to(DEVICE)
    if mid == 19: return BiGRUModel(h).to(DEVICE)
    if mid == 20: return DeepLSTMModel(h).to(DEVICE)
    if mid == 21: return DeepGRUModel(h).to(DEVICE)
    if mid == 22: return CNNGRUModel(h).to(DEVICE)
    if mid == 23: return ResidualLSTMModel(h).to(DEVICE)
    if mid == 24: return AttentionLSTMModel(h).to(DEVICE)
    if mid == 25: return TCNModel(h).to(DEVICE)
    if mid == 26: return CNNAttentionModel(h).to(DEVICE)
    if mid == 27: return ELMModel(input_size=input_size)
    if mid == 28: return DBNModel()
def is_dl(mid):
    return mid >= 13 and mid <= 26

In [ ]:
model_pool = {
    # 0:"Linear",
    # 1:"Ridge",
    # 2:"Lasso",
    # 3:"Huber",
    # 4:"DecisionTree",
    # 5:"RandomForest",
    # 6:"ExtraTrees",
    # 7:"GradientBoosting",
    # 8:"AdaBoost",
    # 9:"SVR",
    # 10:"XGBoost",
    # 11:"LightGBM",
    13:"LSTM",
    14:"GRU",
    15:"BiLSTM",
    16:"CNN",
    17:"CNN_LSTM",

    # 🔷 Attention / Transformer
    18:"Transformer",
    19:"BiGRU",
    20:"Deep_LSTM",
    21:"Deep_GRU",
    22:"CNN_GRU",
    23:"Residual_LSTM",
    24:"Attention_LSTM",
    25:"TCN",
    26:"CNN_Attention",
    27:"ELM"
}

In [17]:
def compute_shap(model, X, model_name):

    try:
        # sample for speed
        if X.shape[0] > 200:
            X = X[np.random.choice(len(X), 200, replace=False)]

        if hasattr(model, "feature_importances_"):
            explainer = shap.TreeExplainer(model)
            shap_vals = explainer.shap_values(X)

        elif hasattr(model, "coef_"):
            explainer = shap.LinearExplainer(model, X)
            shap_vals = explainer.shap_values(X)

        else:
            # skip DL (or use Kernel if needed)
            return None

        return np.abs(shap_vals).mean(axis=0)  # (lag,)

    except:
        return None

In [18]:
def set_seed(seed):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [19]:
def train_model(mid, train_mode, val_mode, test_mode, scaler, lag=lag, horizon=horizon):

    X_train, y_train = create_dataset(train_mode, lag, horizon)

    X_val, y_val = create_dataset(val_mode, lag, horizon)

    X_test, y_test = create_dataset(test_mode, lag, horizon)

    model = build_model(mid, h=horizon,input_size=X_train.shape[1])

    # ---------- ML ----------
    if not is_dl(mid):

        try:
            model.fit(X_train, y_train)
            pred = model.predict(X_test)

        except:
            model.fit(X_train, y_train[:,0])
            p = model.predict(X_test)
            pred = np.tile(p.reshape(-1,1), (1,horizon))

    # ---------- DL ----------
    else:

        model = train_torch(
                            model,
                            X_train,
                            y_train,
                            X_val,
                            y_val,
                            DEVICE,
                            epochs=150,     # increase
                            lr=5e-4,        # slightly lower LR
                            batch_size=64
                        )
        model.eval()

        x = torch.tensor(X_test, dtype=torch.float32)\
                .unsqueeze(-1).to(DEVICE)

        with torch.no_grad():
            pred = model(x).cpu().numpy()

        pred = np.clip(pred, -3, 3)  # StandardScaler domain

    # 🔥 inverse transform
    pred = scaler.inverse_transform(pred.reshape(-1,1)).reshape(pred.shape)
    y    = scaler.inverse_transform(y_test.reshape(-1,1)).reshape(y_test.shape)

    return pred, y, model, X_train

In [20]:
import shap

def compute_shap(model, X, model_name):

    try:
        X_sample = X[:100]

        # =========================
        # TREE MODELS
        # =========================
        if model_name in ["RandomForest","ExtraTrees","XGBoost","LightGBM"]:
            explainer = shap.TreeExplainer(model)
            shap_values = explainer.shap_values(X_sample)
            return np.abs(shap_values)

        # =========================
        # LINEAR MODELS
        # =========================
        elif model_name in ["Linear","Ridge","Lasso"]:
            explainer = shap.LinearExplainer(model, X_sample)
            shap_values = explainer.shap_values(X_sample)
            return np.abs(shap_values)

        # =========================
        # DL MODELS (FIXED)
        # =========================
        else:
            
            torch.backends.cudnn.enabled = False   # 🔥 CRITICAL FIX
            model.train()

            X_torch = torch.tensor(X_sample, dtype=torch.float32)\
                        .unsqueeze(-1).to(DEVICE)

            explainer = shap.GradientExplainer(model, X_torch)
            shap_values = explainer.shap_values(X_torch)

            model.eval()
            torch.backends.cudnn.enabled = True

            return np.abs(shap_values[0])

    except Exception as e:
        print(f"⚠️ SHAP failed for {model_name}: {e}")
        return None

In [21]:
def single_run_model(model_id, run_id):

    set_seed(100 + run_id)
    start = time.time()

    preds, actuals = [], []

    shap_all_modes = []
    mode_importance = []

    for i in range(train_modes_scaled.shape[0]):

        pred, y, model, X_train = train_model(model_id,
            train_modes_scaled[i],val_modes_scaled[i],
            test_modes_scaled[i],
            mode_scalers[i]
        )

        preds.append(pred)
        actuals.append(y)

        # ===============================
        # 🔥 COMPUTE SHAP HERE (FIX)
        # ===============================
        shap_vals = compute_shap(model, X_train, model_pool[model_id])

        if shap_vals is not None:

            # 🔥 reduce to feature importance
            if len(shap_vals.shape) > 1:
                shap_vals = np.mean(np.abs(shap_vals), axis=0)

            shap_all_modes.append(shap_vals)

            # IMF importance
            mode_importance.append(np.mean(shap_vals))

        else:
            shap_all_modes.append(None)
            mode_importance.append(0)

    # ===============================
    # 🔥 RECONSTRUCTION
    # ===============================
    min_len = min(p.shape[0] for p in preds)

    preds   = [p[:min_len] for p in preds]
    actuals = [a[:min_len] for a in actuals]

    final_pred = np.sum(preds, axis=0)
    final_true = np.sum(actuals, axis=0)

    metrics = [
        RMSE(final_true.flatten(), final_pred.flatten()),
        MAE(final_true.flatten(), final_pred.flatten()),
        SMAPE(final_true.flatten(), final_pred.flatten()),
        MASE(final_true.flatten(), final_pred.flatten()),
        R2(final_true.flatten(), final_pred.flatten())
    ]

    runtime = time.time() - start

    print(f"Model: {model_pool[model_id]}, Run: {run_id}, RMSE: {metrics[0]:.4f}, Time: {runtime:.2f}s")

    return final_pred.flatten(), metrics, runtime, shap_all_modes, mode_importance

In [22]:
from joblib import Parallel, delayed
import matplotlib.pyplot as plt

n_jobs = 1 if torch.cuda.is_available() else -1

all_summary = []

for model_id in model_pool.keys():

    print(f"Running Model: {model_pool[model_id]}")

    # 🔥 RESET per model (IMPORTANT FIX)
    all_shap_modes = []
    all_mode_importance = []

    results = Parallel(n_jobs=n_jobs)(
        delayed(single_run_model)(model_id, r) for r in range(RUNS)
    )

    forecasts, metrics, times = [], [], []

    for f, m, t, shap_modes, mode_imp in results:
        forecasts.append(f)
        metrics.append(m)
        times.append(t)

        all_shap_modes.append(shap_modes)
        all_mode_importance.append(mode_imp)

    forecasts = np.array(forecasts)
    metrics   = np.array(metrics)

    name = model_pool[model_id]

    # ===============================
    # 🔥 PATH SETUP
    # ===============================
    base_path = os.path.join(BASE_DIR, f"Bengaluru/results/{a}")
    forecast_dir = os.path.join(base_path, "forecasts")
    metric_dir   = os.path.join(base_path, "metrics")
    log_dir      = os.path.join(base_path, "logs")
    shap_dir     = os.path.join(base_path, "figures/shap")
    imp_dir      = os.path.join(base_path, "figures/feature_importance")

    for d in [forecast_dir, metric_dir, log_dir, shap_dir, imp_dir]:
        os.makedirs(d, exist_ok=True)

    # ===============================
    # 🔷 SAVE FORECASTS
    # ===============================
    pd.DataFrame(forecasts.T).to_csv(
        os.path.join(forecast_dir, f"{name}.csv"),
        index=False
    )

    # ===============================
    # 🔷 SAVE METRICS
    # ===============================
    pd.DataFrame(metrics,
        columns=["RMSE","MAE","SMAPE","MASE","R2"]
    ).to_csv(
        os.path.join(metric_dir, f"{name}.csv"),
        index=False
    )

    # ===============================
    # 🔷 SAVE TIME LOGS
    # ===============================
    pd.DataFrame({
        "run": range(len(times)),
        "time_sec": times
    }).to_csv(
        os.path.join(log_dir, f"{name}.csv"),
        index=False
    )

    # ===============================
    # 🔥 SHAP PROCESSING (FIXED)
    # ===============================
    valid_shap = []

    for run in all_shap_modes:
        for s in run:
            if s is not None:
                s = np.array(s)

                # 🔥 handle (samples, features)
                if len(s.shape) > 1:
                    s = np.mean(np.abs(s), axis=0)

                valid_shap.append(s)

    if len(valid_shap) > 0:

        # 🔥 align lengths
        min_len = min(len(s) for s in valid_shap)
        valid_shap = [s[:min_len] for s in valid_shap]

        combined_shap = np.mean(valid_shap, axis=0)

        plt.figure(figsize=(8,4))
        plt.bar(range(len(combined_shap)), combined_shap)
        plt.title(f"{name} Combined SHAP")
        plt.xlabel("Lag Features")
        plt.ylabel("Mean |SHAP|")

        plt.savefig(os.path.join(shap_dir, f"{name}_combined.png"))
        plt.close()

    else:
        print(f"⚠️ No SHAP for {name}")

    # ===============================
    # 🔥 IMF IMPORTANCE
    # ===============================
    mode_imp = np.array(all_mode_importance)

    if len(mode_imp.shape) > 1:
        mode_imp = np.mean(mode_imp, axis=0)

    plt.figure(figsize=(8,4))
    plt.bar(range(len(mode_imp)), mode_imp)
    plt.title(f"{name} IMF Importance")
    plt.xlabel("IMF Index")
    plt.ylabel("Mean SHAP")

    plt.savefig(os.path.join(imp_dir, f"{name}_IMF_importance.png"))
    plt.close()

    # 🔥 SAVE NUMERIC IMPORTANCE
    pd.DataFrame({
        "IMF": range(len(mode_imp)),
        "Importance": mode_imp
    }).sort_values("Importance", ascending=False).to_csv(
        os.path.join(metric_dir, f"{name}_IMF_importance.csv"),
        index=False
    )

    # ===============================
    # 🔥 SUMMARY
    # ===============================
    mean_vals = np.mean(metrics, axis=0)

    all_summary.append([
        name,
        mean_vals[0],
        mean_vals[1],
        mean_vals[2],
        mean_vals[3],
        mean_vals[4]
    ])


# ===============================
# 🔥 FINAL SUMMARY SAVE
# ===============================
summary_df = pd.DataFrame(all_summary,
    columns=["Model","RMSE","MAE","SMAPE","MASE","R2"]
)

summary_df.to_csv(
    os.path.join(BASE_DIR, f"Bengaluru/results/{a}/metrics", "All_Model_mean_Accuracy.csv"),
    index=False
)

Running Model: CNN_Attention
Model: CNN_Attention, Run: 0, RMSE: 3.2120, Time: 97.28s
Model: CNN_Attention, Run: 1, RMSE: 3.4230, Time: 69.03s
Model: CNN_Attention, Run: 2, RMSE: 3.2048, Time: 136.99s
Model: CNN_Attention, Run: 3, RMSE: 3.3274, Time: 150.89s
Model: CNN_Attention, Run: 4, RMSE: 3.2297, Time: 85.76s
Model: CNN_Attention, Run: 5, RMSE: 3.3018, Time: 70.44s
Model: CNN_Attention, Run: 6, RMSE: 3.3871, Time: 82.00s
Model: CNN_Attention, Run: 7, RMSE: 3.2133, Time: 100.79s
Model: CNN_Attention, Run: 8, RMSE: 3.2588, Time: 91.31s
Model: CNN_Attention, Run: 9, RMSE: 3.1542, Time: 77.70s
Model: CNN_Attention, Run: 10, RMSE: 3.2510, Time: 77.36s
Model: CNN_Attention, Run: 11, RMSE: 3.2428, Time: 78.68s
Model: CNN_Attention, Run: 12, RMSE: 3.3202, Time: 84.04s
Model: CNN_Attention, Run: 13, RMSE: 3.5367, Time: 77.22s
Model: CNN_Attention, Run: 14, RMSE: 3.2482, Time: 77.49s
Model: CNN_Attention, Run: 15, RMSE: 3.1664, Time: 90.77s
Model: CNN_Attention, Run: 16, RMSE: 3.2386, Time:

: 

: 

: 

: 

: 